In [0]:
def pull_user_agg_attributes(spark, db, table_names, run_date, window_size=365, to_pandas=False):
    sdf = spark.sql("""
                    select user_id, 
                    max(months_since_registration) as months_since_registration,
                    max(months_since_consent) as months_since_consent,
                    last(`identity`) as `identity`,
                    last(is_koc) as is_koc,
                    last(has_profile_photo) as has_profile_photo,
                    last(ip_province) as ip_province,
                    last(platform) as platform,
                    last(vehicle_series) as vehicle_series,
                    sum(community_visits) as community_visits,
                    sum(posts_published) as posts_published,
                    sum(posts_viewed) as posts_viewed,
                    sum(posts_liked) as posts_liked,
                    sum(posts_commented) as posts_commented,
                    sum(posts_shared) as posts_shared,
                    sum(users_followed) as users_followed,
                    sum(tribes_joined) as tribes_joined
                    from {db}.{user_attributes}
                    where partition_date between date_sub('{run_date}', {window_size}) and '{run_date}' 
                    group by 1
                    """.format(db = db,
                               user_attributes = table_names['user_attributes'],
                               run_date = run_date,
                               window_size = window_size))
    return sdf.toPandas() if to_pandas else sdf

def pull_post_agg_attributes(spark, db, table_names, run_date, window_size=365, to_pandas=False):
    sdf = spark.sql("""
                    select post_id, 
                    max(days_since_publish) as days_since_published,
                    last(post_type) as post_type,
                    last(is_ugc) as is_ugc,
                    last(is_sink) as is_sink,
                    last(has_pic) as has_pic,
                    last(has_video) as has_video,
                    sum(`views`) as `views`,
                    sum(likes) as likes,
                    sum(comments) as comments,
                    sum(`shares`) as `shares`,
                    sum(dwell_time) as dwell_time 
                    from {db}.{post_attributes}
                    where partition_date between date_sub('{run_date}', {window_size}) and '{run_date}' 
                    group by 1
                    """.format(db = db,
                               post_attributes = table_names["post_attributes"],
                               run_date = run_date,
                               window_size = window_size))
    return sdf.toPandas() if to_pandas else sdf

In [0]:
def retrieve_user_last_n_posts(spark, db, table_names, run_date, last_n=100, weights={'click':1,'like':2,'share':5,'comment':7,'publish':10}, to_pandas=False): 
    sdf = spark.sql("""
                    WITH interaction_degrees AS (
                      SELECT
                        user_id,
                        post_id,
                        interaction,
                        partition_date,
                        CASE
                          WHEN interaction = 'view' THEN {w1}
                          WHEN interaction = 'like' THEN {w2}
                          WHEN interaction = 'share' THEN {w3}
                          WHEN interaction = 'comment' THEN {w4}
                          WHEN interaction = 'publish' THEN {w5}
                          ELSE 0
                        END AS interaction_degree
                      FROM {db}.{user_post_interaction}
                      WHERE partition_date <= '{run_date}'
                      AND interaction != 'impression'
                    ),
                    ranked_posts AS (
                      SELECT
                        *,
                        ROW_NUMBER() OVER (
                          PARTITION BY user_id
                          ORDER BY partition_date DESC
                        ) AS rn
                      FROM interaction_degrees
                    )
                    SELECT
                      user_id,
                      post_id as item_id,
                      SUM(interaction_degree) as interaction
                    FROM ranked_posts
                    WHERE rn <= {last_n}
                    GROUP BY 1, 2
                    """.format(db = db,
                              user_post_interaction = table_names["user_post_interaction"],
                              run_date = run_date,
                              last_n = last_n,
                              w1 = weights['click'],
                              w2 = weights['like'],
                              w3 = weights['share'],
                              w4 = weights['comment'],
                              w5 = weights['publish'])
                  )
    return sdf.toPandas() if to_pandas else sdf
  
def pull_interacted_post_content(spark, retrieved_post_sdf, db, table_names, run_date, to_pandas=False):
    retrieved_post_sdf.createOrReplaceTempView("retrieved_posts")
    sdf = spark.sql("""
                    select rp.user_id, 
                    rp.item_id as post_id, 
                    p.content as text_content, 
                    if(size(mi.pic_url) = 0, NULL, mi.pic_url) as pic_url,
                    if(size(mi.video_url) = 0, NULL, mi.video_url) as video_url
                    from retrieved_posts rp
                    left join datalake_community.dw_cust_community_tb_post p
                    on rp.item_id = p.id
                    and to_date(from_unixtime(p.post_time/1000)) <= '{run_date}'
                    left join (
                      select post_id,
                      collect_list(`url`) filter (where file_type in (0,1,2,3)) as pic_url,
                      collect_list(`url`) filter (where file_type = 10) as video_url
                      from {db}.{media_item_index}
                      group by 1
                    ) mi
                    on rp.item_id = mi.post_id
                    """.format(run_date = run_date,
                               db = db,
                               media_item_index = table_names["media_item_index"]))
    return sdf.toPandas() if to_pandas else sdf

def pull_interacted_post_embedding(spark, retrieved_post_sdf, db, table_names, run_date, collect=False, pool=None, to_pandas=False):
    retrieved_post_sdf.createOrReplaceTempView("retrieved_posts")
    sdf = spark.sql("""
                    select rp.user_id, 
                    pe.post_id, 
                    pe.pooled_embedding as embedding
                    from retrieved_posts rp
                    inner join {db}.{post_embedding} pe
                    on rp.item_id = pe.post_id
                    where pe.partition_date <= '{run_date}'
                    """.format(run_date = run_date,
                               db = db,
                               post_embedding = table_names["post_embedding"]))
    sdf.createOrReplaceTempView("interacted_post_embedding")
    if collect:
        sdf = spark.sql("""
                        select user_id, collect_list(embedding) as embedding_list
                        from interacted_post_embedding
                        group by 1
                        """)
    if pool == 'sum':
        # Element-wise sum pooling for embeddings
        sdf = spark.sql("""              
                        SELECT
                          user_id,
                          AGGREGATE(
                            COLLECT_LIST(embedding),
                            ARRAY_REPEAT(CAST(0.0 AS DOUBLE), MAX(SIZE(embedding))),
                            (acc, x) -> TRANSFORM(
                              ZIP_WITH(acc, x, (a, b) -> a + b),
                              y -> y
                            )
                          ) AS pooled_embedding
                        FROM interacted_post_embedding
                        GROUP BY user_id
                        """)
    elif pool == 'avg':
        # Element-wise average pooling for embeddings
        sdf = spark.sql("""
                        SELECT
                          user_id,
                          TRANSFORM(
                            AGGREGATE(
                              COLLECT_LIST(embedding),
                              ARRAY_REPEAT(CAST(0.0 AS DOUBLE), MAX(SIZE(embedding))),
                              (acc, x) -> TRANSFORM(
                                ZIP_WITH(acc, x, (a, b) -> a + b),
                                y -> y
                              )
                            ),
                            x -> x / COUNT(embedding)
                          ) AS pooled_embedding
                        FROM interacted_post_embedding
                        GROUP BY user_id
                        """)
      
    return sdf.toPandas() if to_pandas else sdf

In [0]:
def write_user_feature_table(spark, run_date, db, table_names, view_names, emb_col='pooled_embedding'):
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{user_features} PARTITION (partition_date = '{run_date}')
              select ua.*, lpe.{emb_col} as lastn_embs
              from {user_attributes} ua
              left join {lastn_post_embedding} lpe
              on ua.user_id = lpe.user_id
              """.format(db = db,
                          user_features = table_names["user_features"],
                          user_attributes = view_names["user_attributes"],
                          lastn_post_embedding = view_names["lastn_post_embedding"],
                          emb_col = emb_col,
                          run_date = run_date))

def write_post_feature_table(spark, run_date, db, table_names, view_names):     
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{post_features} PARTITION (partition_date = '{run_date}')
              select pa.*, pe.pooled_embedding as post_content_emb
              from {post_attributes} pa
              left join {db}.{post_embedding} pe
              on pa.post_id = pe.post_id
              """.format(run_date = run_date,
                          db = db,
                          post_features = table_names["post_features"],
                          post_embedding = table_names["post_embedding"],
                          post_attributes = view_names["post_attributes"]))     
    
def write_user_post_aggregated_interaction_table(spark, run_date, db, table_names, weights={'click':1,'like':2,'share':5,'comment':7,'publish':10}):
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{user_post_interaction_agg} PARTITION (partition_date = '{run_date}')
              SELECT user_id,
              post_id,
              SUM(CASE
                  WHEN interaction = 'view' THEN {w1}
                  WHEN interaction = 'like' THEN {w2}
                  WHEN interaction = 'share' THEN {w3}
                  WHEN interaction = 'comment' THEN {w4}
                  WHEN interaction = 'publish' THEN {w5}
                  ELSE 0
              END) AS interaction
              FROM {db}.{user_post_interaction}
              WHERE partition_date = '{run_date}'
              AND user_id in (select user_id from {db}.{user_features} where partition_date = '{run_date}')
              AND post_id in (select post_id from {db}.{post_features} where partition_date = '{run_date}')
              GROUP BY 1, 2
              """.format(run_date = run_date,
                          db = db,
                          user_post_interaction = table_names["user_post_interaction"],
                          user_post_interaction_agg = table_names["user_post_interaction_agg"],
                          user_features = table_names["user_features"],
                          post_features = table_names["post_features"],
                          w1 = weights['click'],
                          w2 = weights['like'],
                          w3 = weights['share'],
                          w4 = weights['comment'],
                          w5 = weights['publish']))

Concatenate features

In [0]:
import json

# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

In [0]:
view_names = model_config['VIEW_NAMES']
window_size = model_config['FEATURE_PROCESSING_PARAMS']['feature_agg_window_size']
last_n = model_config['FEATURE_PROCESSING_PARAMS']['last_n_embeddings']
weights = model_config['FEATURE_PROCESSING_PARAMS']['interaction_weights']
emb_pooling = model_config['FEATURE_PROCESSING_PARAMS']['emb_pooling']

In [0]:
user_attr_sdf = pull_user_agg_attributes(spark, db, table_names, run_date, window_size)
post_attr_sdf = pull_post_agg_attributes(spark, db, table_names, run_date, window_size)
lastn_post_sdf = retrieve_user_last_n_posts(spark, db, table_names, run_date, last_n, weights)
lastn_embedding_sdf = pull_interacted_post_embedding(spark, lastn_post_sdf, db, table_names, run_date, pool=emb_pooling)

user_attr_sdf.createOrReplaceTempView(view_names["user_attributes"])
post_attr_sdf.createOrReplaceTempView(view_names["post_attributes"])
lastn_embedding_sdf.createOrReplaceTempView(view_names["lastn_post_embedding"])

write_user_feature_table(spark, run_date, db, table_names, view_names, emb_col='pooled_embedding')
write_post_feature_table(spark, run_date, db, table_names, view_names)
write_user_post_aggregated_interaction_table(spark, run_date, db, table_names, weights)